# Finnish Financial Sent Template Notebook

Compatible with Sentence-Transformers (SBERT) type models, utlizing SetFit.

## Install

In [1]:
!pip install -q "setfit" "transformers==4.44.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 137.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 155.6 MB/s eta 0:00:00


## Imports

In [2]:
import warnings

import os
os.environ["WANDB_DISABLED"] = "true"

from datetime import datetime
import gc
import psutil

from google.colab import drive

from setfit import SetFitModel, Trainer as SetFitTrainer, TrainingArguments as SetFitArgs

from sklearn.model_selection import StratifiedKFold, train_test_split

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

from datasets import Dataset
import pandas as pd
import datetime
import numpy as np
import torch

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Mount Google Drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


## GPU/RAM Check

In [4]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sun Mar 22 11:22:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 189.9 gigabytes of available RAM

You are using a high-RAM runtime!


In [6]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
print(f"Current timestamp: {timestamp}")

Current timestamp: 2026-03-22_11-22-39


## Load Model

In [7]:
BASE_MODEL = 'intfloat/multilingual-e5-large'
NUM_LABELS  = 3
LABEL_NAMES = ["negative", "neutral", "positive"]

print(f"Base model: {BASE_MODEL}")

Base model: intfloat/multilingual-e5-large


## Dataset Helpers

In [8]:
def make_setfit_dataset(df, text_col="text", label_col="label"):
    """Plain text+label dataset for SetFit (no tokenization needed)."""
    return Dataset.from_dict({
        "text":  df[text_col].tolist(),
        "label": df[label_col].tolist(),
    })

def make_forum_dataset(df):
    """Format forum posts (company_name + message) into SetFit dataset."""
    df = df.copy()
    df["text"] = "yritys: " + df["company_name"] + " viesti: " + df["message"]
    return make_setfit_dataset(df, text_col="text", label_col="sentiment")

## Load Human-labeled Financial Data

In [9]:
human_labeled_sentiment = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/labeled.parquet')

# Replace with your dataset loading
ds = human_labeled_sentiment

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
ds.sample(5)

,id,author_id,message,date_time,engagement,forum,url,company_name,ticker,message_length,year_month,year,sentiment,labeled_at
77,Kauppalehti.post-7616837,Kauppalehti.57519,Arvuuttelija70 sanoi:\nTämä on tällaista... os...,2025-08-26 12:12:27.000,"Reactions:\nVerot, öölman ja Arvuuttelija70",Kauppalehti,https://keskustelu.kauppalehti.fi/threads/endo...,Endomines Finland,PAMPALO,661,2025-08,2025,1,2026-03-16T14:58:37.056246
134,Sijoitustieto.comment-110091,Sijoitustieto.Unknown,"Olettaen, että 85k on samalta myyjältä, suurim...",2024-07-29 15:58:00.000,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Investors House,INVEST,172,2024-07,2024,1,2026-03-16T17:24:57.008139
161,Inderes.312900,Inderes.14936,Yhtiöstä tehdyistä kommenteista paistaa läpi p...,2021-07-02 14:12:47.098,13,Inderes,https://forum.inderes.com/t/admicom-rakennusal...,Admicom,ADMCM,795,2021-07,2021,2,2026-03-16T17:35:18.601303
407,Inderes.143457,Inderes.35,"Sammon IR blogissa juttua, miten bondien liikk...",2020-09-22 06:24:16.586,22,Inderes,https://forum.inderes.com/t/sampo-vakuuttava-v...,Sampo,SAMPO,268,2020-09,2020,2,2026-03-16T19:58:04.123875
593,Sijoitustieto.comment-21683,Sijoitustieto.Unknown,"Mie lisäsin Nordeaa @8,78. Poikkeuksellisesti ...",2018-10-04 11:29:00.000,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Nordea,NDA-FI,331,2018-10,2018,2,2026-03-16T21:02:01.323939


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
ds["sentiment"].value_counts()

,count
sentiment,
1,253
2,205
0,150


## K-Fold Cross Validation (Final Phase)

In [12]:
N_FOLDS = 5

cv_df = ds[["id", "message", "sentiment", "company_name"]].reset_index(drop=True)

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

_label_map = {name: i for i, name in enumerate(LABEL_NAMES)}

def to_int_preds(raw):
    arr = list(raw)
    if arr and isinstance(arr[0], str):
        return np.array([_label_map[p] for p in arr])
    return np.array(arr)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(cv_df, cv_df["sentiment"])):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*55}")

    fold_train_df = cv_df.iloc[train_idx][["message", "sentiment", "company_name"]]
    fold_val_df   = cv_df.iloc[val_idx][["message", "sentiment", "company_name"]]
    fold_true     = fold_val_df["sentiment"].tolist()

    print(f"Train: {len(fold_train_df)}  |  Val: {len(fold_val_df)}")

    fold_train_ds = make_forum_dataset(fold_train_df)
    fold_val_ds   = make_forum_dataset(fold_val_df)

    fold_model = SetFitModel.from_pretrained(BASE_MODEL, num_classes=NUM_LABELS)

    fold_args = SetFitArgs(
        output_dir=f'/tmp/fold_{fold+1}_setfit/',
        num_epochs=1,
        num_iterations=10,
        body_learning_rate=2e-5,
        head_learning_rate=1e-2,
        batch_size=16,
        report_to="none",
    )

    fold_trainer = SetFitTrainer(
        model=fold_model,
        args=fold_args,
        train_dataset=fold_train_ds,
        eval_dataset=fold_val_ds,
        metric="f1",
        metric_kwargs={"average": "macro"},
    )
    fold_trainer.train()

    fold_preds = to_int_preds(fold_model.predict([row["text"] for row in fold_val_ds]))

    fold_results.append({
        "accuracy":    accuracy_score(fold_true, fold_preds),
        "f1_weighted": f1_score(fold_true, fold_preds, average="weighted", zero_division=0),
        "f1_macro":    f1_score(fold_true, fold_preds, average="macro",    zero_division=0),
        "report":      classification_report(fold_true, fold_preds, target_names=LABEL_NAMES, output_dict=True, zero_division=0),
        "confusion":   confusion_matrix(fold_true, fold_preds),
    })
    print(f"  acc={fold_results[-1]['accuracy']:.3f}  f1_weighted={fold_results[-1]['f1_weighted']:.3f}  f1_macro={fold_results[-1]['f1_macro']:.3f}")

    del fold_model, fold_trainer
    gc.collect(); torch.cuda.empty_cache()


  FOLD 1/5
Train: 486  |  Val: 122


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

***** Running training *****
  Num unique pairs = 4870
  Batch size = 16
  Num epochs = 2
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)


Step,Training Loss
1,0.158800
50,0.285000
100,0.251300
150,0.243700
200,0.230600
250,0.195000
300,0.131500
350,0.074200
400,0.058600
450,0.041600


/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: Deprecati

  acc=0.579  f1_weighted=0.579  f1_macro=0.581


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:
accs = [r["accuracy"]    for r in fold_results]
f1w  = [r["f1_weighted"] for r in fold_results]
f1m  = [r["f1_macro"]    for r in fold_results]

print("── Per-fold Results ──")
for i, r in enumerate(fold_results):
    print(f"  Fold {i+1}: acc={r['accuracy']:.3f}  f1_weighted={r['f1_weighted']:.3f}  f1_macro={r['f1_macro']:.3f}")

print(f"\n── Mean ± Std ──")
print(f"  Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"  F1 weighted : {np.mean(f1w):.3f} ± {np.std(f1w):.3f}")
print(f"  F1 macro    : {np.mean(f1m):.3f} ± {np.std(f1m):.3f}")

print(f"\n── Mean Per-class Metrics (across folds) ──")
for cls in LABEL_NAMES:
    prec = np.mean([r["report"][cls]["precision"] for r in fold_results])
    rec  = np.mean([r["report"][cls]["recall"]    for r in fold_results])
    f1   = np.mean([r["report"][cls]["f1-score"]  for r in fold_results])
    print(f"  {cls:>10}: precision={prec:.3f}  recall={rec:.3f}  f1={f1:.3f}")

agg_cm = sum(r["confusion"] for r in fold_results)
print(f"\n── Aggregated Confusion Matrix (all folds) ──")
print(pd.DataFrame(
    agg_cm,
    index=[f"true_{l}"  for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

── Per-fold Results ──
  Fold 1: acc=0.484  f1_weighted=0.490  f1_macro=0.482
  Fold 2: acc=0.557  f1_weighted=0.563  f1_macro=0.547
  Fold 3: acc=0.566  f1_weighted=0.565  f1_macro=0.558
  Fold 4: acc=0.620  f1_weighted=0.618  f1_macro=0.601
  Fold 5: acc=0.579  f1_weighted=0.579  f1_macro=0.581

── Mean ± Std ──
  Accuracy    : 0.561 ± 0.044
  F1 weighted : 0.563 ± 0.042
  F1 macro    : 0.554 ± 0.040

── Mean Per-class Metrics (across folds) ──
    negative: precision=0.441  recall=0.540  f1=0.480
     neutral: precision=0.622  recall=0.562  f1=0.587
    positive: precision=0.628  recall=0.576  f1=0.595

── Aggregated Confusion Matrix (all folds) ──
               pred_negative  pred_neutral  pred_positive
true_negative             81            43             26
true_neutral              66           142             45
true_positive             44            43            118


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Final Model (All Data)

In [ ]:
FINAL_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/models/{BASE_MODEL.split("/")[-1]}_final_{timestamp}/'

all_human_df   = ds[["message", "sentiment", "company_name"]].copy()
final_train_ds = make_forum_dataset(all_human_df)

print(f"Final fine-tuning on {len(all_human_df):,} human-labeled samples")
print(all_human_df["sentiment"].value_counts().sort_index().rename({0: "negative", 1: "neutral", 2: "positive"}))

final_model = SetFitModel.from_pretrained(BASE_MODEL, num_classes=NUM_LABELS)

final_args = SetFitArgs(
    output_dir='/tmp/final_setfit/',
    num_epochs=1,
    num_iterations=5,
    body_learning_rate=2e-4,
    head_learning_rate=1e-2,
    batch_size=16,
    report_to="none",
)

_final_start = datetime.datetime.now()

final_trainer = SetFitTrainer(
    model=final_model,
    args=final_args,
    train_dataset=final_train_ds,
    metric="f1",
    metric_kwargs={"average": "macro"},
)
final_trainer.train()

_final_elapsed = datetime.datetime.now() - _final_start
print(f"Pipeline runtime: {str(_final_elapsed).split('.')[0]}")

final_model.save_pretrained(FINAL_MODEL_PATH)
print(f"Final model saved to: {FINAL_MODEL_PATH}")

Final fine-tuning on 608 human-labeled samples
sentiment
negative    150
neutral     253
positive    205
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime

Map:   0%|          | 0/608 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 6080
  Batch size = 16
  Num epochs = 1
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)


Step,Training Loss
1,0.236100
50,0.321200
100,0.302100
150,0.311000
200,0.303100
250,0.283300
300,0.268900
350,0.261400


KeyboardInterrupt: 

/usr/local/lib/python3.12/dist-packages/IPython/core/history.py:576: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  self.db.execute("""UPDATE sessions SET end=?, num_cmds=? WHERE
